In [1]:
import glob, time, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.decomposition import PCA
NB = glob.glob('/kaggle/input/notebooks/aadiparkhi/03a*')[0]
BK, K = ['dino', 'convnext'], 32
rng = np.random.default_rng(0); torch.manual_seed(0)
D = {}
for d in ['places', 'bdd']:
    z = np.load(f'{NB}/feats_{d}.npz'); m = pd.read_csv(f'{NB}/meta_{d}.csv')
    D[d] = {'meta': m, **{k: {'emb': z[f'{k}_emb'].astype(np.float32), 'log': z[f'{k}_log']} for k in BK}}

def ks_stat(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0).mean()

def train_ae(z, fit):
    x = torch.tensor(z, dtype=torch.float32); tr = x[torch.tensor(fit)]
    ae = nn.Sequential(nn.Linear(x.shape[1], 256), nn.ReLU(), nn.Linear(256, 32), nn.ReLU(),
                       nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, x.shape[1]))
    opt = torch.optim.Adam(ae.parameters(), 1e-3)
    for _ in range(60):
        for b in torch.randperm(len(tr)).split(256):
            opt.zero_grad(); ((ae(tr[b]) - tr[b]) ** 2).mean().backward(); opt.step()
    with torch.no_grad(): return ((ae(x) - x) ** 2).mean(1).numpy()

def prep(d, k):
    m, e = D[d]['meta'], D[d][k]['emb']; v = (m.set == 'val').values
    z = (e - e[v].mean(0)) / (e[v].std(0) + 1e-6)
    P = PCA(K, random_state=0).fit(z[v]).transform(z)
    return P, np.sort(P[v], 0), train_ae(z, m.set.isin(['val', 'train']).values)

def batches(rows, B, nb): return [rng.choice(rows, B) for _ in range(nb)]

t = time.time()
PP = {(d, k): prep(d, k) for d in D for k in BK}
print({d: len(D[d]['meta']) for d in D}, round(time.time() - t), 's')

{'places': 36984, 'bdd': 36046} 46 s


In [2]:
def correct(d, k):
    m, L = D[d]['meta'], D[d][k]['log']
    return (L.argmax(1) == m.label.values) if d == 'places' else (L[:, :11].argmax(1) == m.cond_id.values)

def sc(d, b):
    out = []
    for k in BK:
        P, S, err = PP[(d, k)]; out += [ks_stat(P[b], S), err[b].mean()]
    return np.array(out)

CFG = {'dino': [0, 1], 'convnext': [2, 3], 'both': [0, 1, 2, 3]}

def two(d, B, nb=10):
    m = D[d]['meta']; ok = correct(d, 'dino')
    cp = np.where((m.set == 'corrupt').values)[0]; bid, sev, cor = m.bid.values, m.severity.values, m.corruption.values
    i0 = cp[sev[cp] == 0]; ck = np.zeros(bid[cp].max() + 1); ck[bid[i0]] = ok[i0]
    ns = lambda pool, n: np.array([sc(d, b) for b in batches(pool, B, n)])
    tst = np.where((m.set == 'test').values)[0]; rng.shuffle(tst); h = len(tst) // 2
    Ncal, Nevl, N1, N2 = ns(tst[:h], 400), ns(tst[h:], 400), ns(i0, 400), ns(i0, 400)
    cn = sorted(set(cor[cp]) - {'clean'}); X, drop = {}, {}
    for c in cn:
        for s in range(1, 6):
            pool = cp[(cor[cp] == c) & (sev[cp] == s)]
            X[(c, s)] = ns(pool, nb); drop[(c, s)] = (ck[bid[pool]] - ok[pool]).mean()
    rows, pc = [], {}
    for name, cl in CFG.items():
        q = 1 - 0.01 / len(cl)                                   # same 1% total false-alarm budget for every setup
        tp, tf = np.quantile(N1[:, cl], q, axis=0), np.quantile(Ncal[:, cl], q, axis=0)
        fire = lambda A, t: (A[:, cl] > t).any(1)
        det = {key: fire(v, tp).mean() for key, v in X.items()}
        rows.append(dict(domain=d, B=B, setup=name, far_paired=fire(N2, tp).mean(), far_fresh=fire(Nevl, tf).mean(),
                         sev1=np.mean([v for (c, s), v in det.items() if s == 1]), sev2=np.mean([v for (c, s), v in det.items() if s == 2]),
                         harmful=np.mean([v for key, v in det.items() if drop[key] >= 0.05] or [np.nan]),
                         low_impact=np.mean([v for key, v in det.items() if drop[key] < 0.05] or [np.nan])))
        pc[name] = pd.Series({c: det[(c, 1)] for c in cn})
    return pd.DataFrame(rows), pd.DataFrame(pc)

for d in D:
    for B in (50, 100):
        r, pc = two(d, B)
        print(r.round(3).to_string(index=False))
        if B == 100: print(pc.round(2).to_string())

domain  B    setup  far_paired  far_fresh  sev1  sev2  harmful  low_impact
places 50     dino       0.020      0.018 0.713 0.800    0.978       0.300
places 50 convnext       0.012      0.010 0.813 0.873    0.986       0.533
places 50     both       0.025      0.025 0.833 0.867    0.997       0.550
domain   B    setup  far_paired  far_fresh  sev1  sev2  harmful  low_impact
places 100     dino       0.020      0.022 0.800 0.880    0.998       0.533
places 100 convnext       0.008      0.010 0.853 0.920    1.000       0.700
places 100     both       0.022      0.020 0.860 0.947    1.000       0.750
                   dino  convnext  both
brightness          0.0       0.0   0.0
contrast            0.4       0.5   0.6
defocus_blur        1.0       1.0   1.0
elastic_transform   1.0       1.0   1.0
fog                 1.0       1.0   1.0
frost               1.0       1.0   1.0
gaussian_noise      0.9       1.0   1.0
glass_blur          1.0       1.0   1.0
impulse_noise       1.0       1.0   

In [3]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

def ks_vec(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0)

def build(d, B=100, nb=30, seed=0):
    r = np.random.default_rng(seed); m = D[d]['meta']; ok = correct(d, 'dino')
    cp = np.where((m.set == 'corrupt').values)[0]; bid, sev, cor = m.bid.values, m.severity.values, m.corruption.values
    i0 = cp[sev[cp] == 0]; ck = np.zeros(bid[cp].max() + 1); ck[bid[i0]] = ok[i0]
    half = r.permutation(bid[cp].max() + 1) % 2                      # two disjoint halves of the base images
    F = {k: [] for k in BK}; meta = []
    for c in sorted(set(cor[cp])):
        for s in sorted(set(sev[cp][cor[cp] == c])):
            pool = cp[(cor[cp] == c) & (sev[cp] == s)]
            for h in (0, 1):
                ph = pool[half[bid[pool]] == h]
                for _ in range(nb):
                    b = r.choice(ph, B)
                    for k in BK:
                        P, S, err = PP[(d, k)]; X = P[b]; kv = ks_vec(X, S)
                        F[k].append(np.concatenate([X.mean(0), X.std(0), kv, [kv.mean(), err[b].mean()]]))
                    meta.append((c, s, h, ck[bid[b]].mean() - ok[b].mean()))
    return {k: np.array(v) for k, v in F.items()}, pd.DataFrame(meta, columns=['cause', 'sev', 'split', 'dloss'])

rows = []
for d in D:
    for seed in (1, 2, 3):
        F, mt = build(d, seed=seed)
        tr, te = (mt.split == 0).values, (mt.split == 1).values
        mu = mt[tr].groupby(['cause', 'sev']).dloss.mean()                      # oracle: knows the true condition
        o = np.array([mu[(c, s)] for c, s in zip(mt.cause[te], mt.sev[te])])
        rows.append(dict(domain=d, seed=seed, feats='oracle', r2=r2_score(mt.dloss[te], o), mae=mean_absolute_error(mt.dloss[te], o)))
        for name, X in {'dino': F['dino'], 'convnext': F['convnext'], 'both': np.hstack([F['dino'], F['convnext']])}.items():
            a, b = tr & (mt.cause != 'clean').values, te & (mt.cause != 'clean').values
            clf = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X[a], mt.cause[a])
            p = RandomForestRegressor(300, random_state=0, n_jobs=-1).fit(X[tr], mt.dloss[tr]).predict(X[te])
            rows.append(dict(domain=d, seed=seed, feats=name, attr=(clf.predict(X[b]) == mt.cause[b]).mean(), r2=r2_score(mt.dloss[te], p), mae=mean_absolute_error(mt.dloss[te], p)))
print(pd.DataFrame(rows).drop(columns='seed').groupby(['domain', 'feats']).agg(['mean', 'std']).round(3).to_string())

                    r2           mae          attr       
                  mean    std   mean    std   mean    std
domain feats                                             
bdd    both      0.798  0.035  0.056  0.007  0.995  0.003
       convnext  0.771  0.046  0.060  0.008  0.990  0.002
       dino      0.804  0.033  0.055  0.007  0.990  0.001
       oracle    0.788  0.041  0.057  0.008    NaN    NaN
places both      0.857  0.055  0.055  0.011  0.971  0.002
       convnext  0.825  0.064  0.060  0.010  0.937  0.004
       dino      0.864  0.048  0.054  0.010  0.935  0.012
       oracle    0.853  0.067  0.056  0.013    NaN    NaN


In [4]:
NAMES = ['clear_day', 'clear_night', 'clear_twilight', 'overcast_day', 'overcast_twilight', 'rainy_day', 'rainy_night',
         'rainy_twilight', 'snow_fog_day', 'snow_fog_night', 'snow_fog_twilight']
GROUPS = {'night': lambda n: n.endswith('night'), 'day': lambda n: n.endswith('_day'), 'twilight': lambda n: n.endswith('twilight'),
          'rain': lambda n: n.startswith('rainy'), 'snow_fog': lambda n: n.startswith('snow_fog'), 'overcast': lambda n: n.startswith('overcast')}
d = 'bdd'; m = D[d]['meta']; cond = m.cond_id.values; ok = correct(d, 'dino')
pref = np.bincount(cond[(m.set == 'val').values], minlength=11) / (m.set == 'val').sum()   # reference mix = clean val
tst = np.where((m.set == 'test').values)[0]; rng.shuffle(tst); h = len(tst) // 2
PA, PB = tst[:h], tst[h:]

def draw(pool, p, B=100):
    cl = rng.choice(11, B, p=p); idx = np.empty(B, int)
    for c in range(11):
        w = np.where(cl == c)[0]
        if len(w): idx[w] = rng.choice(pool[cond[pool] == c], len(w))
    return idx

def run(pool, p, n):
    bs = [draw(pool, p) for _ in range(n)]
    return np.array([sc(d, b) for b in bs]), np.array([ok[b].mean() for b in bs])

Ncal, _ = run(PA, pref, 400); Nevl, acc0 = run(PB, pref, 400)
thr = {k: np.quantile(Ncal[:, c], 1 - 0.01 / len(c), axis=0) for k, c in CFG.items()}
fire = lambda A, k: (A[:, CFG[k]] > thr[k]).any(1).mean()
print('false alarms (reference mix):', {k: round(fire(Nevl, k), 3) for k in CFG}, ' DINOv2 batch accuracy:', round(acc0.mean(), 3))
rows = []
for g, f in GROUPS.items():
    u = np.array([float(f(n)) for n in NAMES]); u /= u.sum()
    for lam in (0.25, 0.5, 0.75):
        X, acc = run(PB, (1 - lam) * pref + lam * u, 30)
        rows.append(dict(group=g, lam=lam, **{k: fire(X, k) for k in CFG}, acc_change=acc.mean() - acc0.mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))

false alarms (reference mix): {'dino': np.float64(0.002), 'convnext': np.float64(0.015), 'both': np.float64(0.015)}  DINOv2 batch accuracy: 0.726
   group  lam  dino  convnext  both  acc_change
   night 0.25 0.033     0.100 0.100       0.025
   night 0.50 0.633     0.367 0.667       0.046
   night 0.75 1.000     1.000 1.000       0.068
     day 0.25 0.033     0.033 0.033       0.005
     day 0.50 0.100     0.267 0.200      -0.001
     day 0.75 0.633     0.733 0.833       0.008
twilight 0.25 0.033     0.100 0.067      -0.046
twilight 0.50 0.433     0.600 0.567      -0.118
twilight 0.75 1.000     1.000 1.000      -0.161
    rain 0.25 0.100     0.067 0.067      -0.040
    rain 0.50 0.700     0.767 0.833      -0.072
    rain 0.75 1.000     1.000 1.000      -0.122
snow_fog 0.25 0.033     0.000 0.033       0.012
snow_fog 0.50 0.533     0.433 0.600       0.011
snow_fog 0.75 1.000     1.000 1.000       0.047
overcast 0.25 0.000     0.167 0.067      -0.029
overcast 0.50 0.900     1.000 0.967   